## Imports

In [2]:
import cv2
import numpy as np
from pathlib import Path
import time
from challenge.utils.camera import CameraDisplay

from helperpost import npFilter_boxes, npNms, npDisplayBoxes
from helpertrt import do_inference_v2, allocate_buffers, get_engine

ModuleNotFoundError: No module named 'jetcam'

## load TinyYOLO

In [4]:
# load onnx model
t0 = time.time()
onnx_path = 'models/person_only_baseline/model_best_bnopt.onnx'#'model_best_bnopt-sim.onnx'#
trt_path = 'models/person_only_baseline/modelv1.trt' # 'modelenginev2.trt'
engine = get_engine(onnx_path, trt_path) # if trt file exists, then use it, else create engine from onnx file
print(f"load engine: \t {(time.time() - t0)*1000:.2f} ms") 

# create context
t0 = time.time()
context = engine.create_execution_context()
print(f"create context: \t {(time.time() - t0)*1000:.2f} ms") 

# allocate memory in GPU for the model
t0 = time.time()
inputs, outputs, bindings, stream = allocate_buffers(engine)
print(f"alloc buf: \t {(time.time() - t0)*1000:.2f} ms") 
    

TinyYoloV2(
  (pad): ReflectionPad2d((0, 1, 0, 1))
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv5): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn5): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv6): Conv2d(256, 512, kernel_size=(3, 3), stride=

## callback

In [ ]:
normalization_factor = 1 / 255.0

In [7]:
def callback(frame):
    global now

    fps = f"{int(1/(time.time() - now))}"
    now = time.time()
        
    frame = frame.astype('float32') 
        
    # model requires 1x3x320x320 as input
    #test = time.time()
    inputs[0].host = np.asarray(frame * normalization_factor).transpose(2,0,1).reshape(1,3,320,320).astype('float16')
    trt_output = do_inference_v2(context, bindings=bindings, inputs=inputs, outputs=outputs, stream=stream)
    # trt output is a list with 3000 values 
    # expected output shape is (1, 1, 5, 10, 10, 6)
    output = np.asarray(trt_output).reshape((1, 1, 5, 10, 10, 6))     
    
    # post-processing
    # filter boxes based on confidence score (class_score*confidence)
    output = npFilter_boxes(output[0], 0.1)
    #filter boxes based on overlap
    output = npNms(output, 0.25)
    # add bounding boxes
    frame = npDisplayBoxes(frame, output)
    #print(f"model: \t {(time.time() - test)*1000:.2f} ms")
    
    # display frame rate
    cv2.putText(frame, "fps="+fps, (2, 25), cv2.FONT_HERSHEY_SIMPLEX, 1,
                (100, 255, 0), 2, cv2.LINE_AA)
    
    return frame

## Camera loop

In [ ]:
# Initialize the camera with the callback
cam = CameraDisplay(callback)

In [ ]:
# The camera stream can be started with cam.start()
# The callback gets asynchronously called (can be stopped with cam.stop())
cam.start()

In [ ]:
# The camera should always be stopped and released for a new camera is instantiated (calling CameraDisplay(callback) again)
cam.stop()
cam.release()